In [ ]:
from pathlib import Path
import json
import urllib.request
import numpy as np
import pandas as pd
from IPython.display import display
import subprocess
import plotly.graph_objects as go

# Parameters to tune
START_DATE = "2026-06-30"
END_DATE = "2026-07-28"

UNCORRELATED_BPS_BY_CHAIN = {
    "ethereum": 4.0, "gnosis": 3.0, "arbitrum": 1.0, "base": 2.0,
    "avalanche_c": 2.0, "polygon": 3.0, "bnb": 1.0,
}
CORRELATED_BPS = 0.1

# Constants
TOKEN_LISTS_URL = "https://cms.cow.finance/api/correlated-tokens?pagination[pageSize]=100"
WEI = 1e18
DATA_DIR = Path("../data")
TOKEN_LISTS_PATH = DATA_DIR / "token_lists.json"
CHAIN_ALIASES = {
    "ethereum": ("mainnet", "ethereum"), "gnosis": ("gnosis", "xdai"),
    "arbitrum": ("arbitrum",), "base": ("base",), "polygon": ("polygon",),
    "bnb": ("bnb", "bsc"), "avalanche_c": ("avalanche",),
}
FETCH_SCRIPT = Path("../scripts/fetch_data.py")

AUCTION_KEYS = ["blockchain", "auction_id", "solver"]
PERIOD_KEYS = ["blockchain", "accounting_period", "solver"]


def input_paths(data_dir, chain, start, end):
    """The three CSVs scripts/fetch_data.py writes per chain and window."""
    data_dir = Path(data_dir)
    return {
        "rewards": data_dir / f"{chain}_{start}_{end}.csv",
        "failed volumes": data_dir / f"{chain}_{start}_{end}_failed_volumes.csv",
        "consistency shares": data_dir / f"{chain}_{start}_{end}_consistency_shares.csv",
    }


def fetch_inputs_if_missing(
    start_date,
    end_date,
    chains,
    data_dir=DATA_DIR,
    fetch_script=FETCH_SCRIPT,
):
    data_dir = Path(data_dir)
    fetch_script = Path(fetch_script)
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    data_dir.mkdir(parents=True, exist_ok=True)
    if not fetch_script.exists():
        raise FileNotFoundError(f"Fetch script not found: {fetch_script}")
    for chain in chains:
        paths = input_paths(data_dir, chain, start, end)
        if all(path.exists() for path in paths.values()):
            print(f"{chain}: using cached CSVs for {start} to {end}")
            continue
        print(f"{chain}: fetching data for {start} to {end}")
        subprocess.run(
            ["uv", "run", "python", str(fetch_script), "--chain", chain,
             "--start", start, "--end", end, "--out", str(paths["rewards"])],
            check=True,
        )
        for label, path in paths.items():
            if not path.exists():
                raise FileNotFoundError(
                    f"Fetch completed but the {label} CSV was not created: {path}"
                )


def load_inputs(data_dir, start_date, end_date, chains):
    data_dir = Path(data_dir)
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")

    paths_by_chain = {
        chain: input_paths(data_dir, chain, start, end) for chain in chains
    }
    missing = [
        path
        for paths in paths_by_chain.values()
        for path in paths.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing input CSVs:\n" + "\n".join(str(path) for path in missing)
        )

    def read(label):
        print(f"{label}:")
        for paths in paths_by_chain.values():
            print(" ", paths[label].name)
        return pd.concat(
            [
                pd.read_csv(paths[label], low_memory=False)
                for paths in paths_by_chain.values()
            ],
            ignore_index=True,
        ).drop_duplicates()

    return read("rewards"), read("failed volumes"), read("consistency shares")


def load_correlated_groups(chains):
    if not TOKEN_LISTS_PATH.exists():
        TOKEN_LISTS_PATH.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(TOKEN_LISTS_URL) as response:
            TOKEN_LISTS_PATH.write_bytes(response.read())

    payload = json.loads(TOKEN_LISTS_PATH.read_text())

    named_groups = [
        (
            entry["attributes"]["name"].lower(),
            {str(token).lower() for token in entry["attributes"]["tokens"]},
        )
        for entry in payload["data"]
    ]

    groups_by_chain = {}
    for chain in chains:
        aliases = CHAIN_ALIASES[chain]
        groups = [
            tokens for name, tokens in named_groups
            if any(alias in name for alias in aliases)
        ]
        if not groups:
            raise ValueError(f"No correlated-token groups found for {chain}")
        groups_by_chain[chain] = groups

    return groups_by_chain


def proposed_penalty_caps(volumes, groups_by_chain):
    """The proposed cap per (auction, solver): rate x failed volume.

    The rate depends on the token pair, so it is applied per pair and then
    summed. Volumes stay inside this function -- what the counterfactual needs
    downstream is the cap itself, not the volumes it was derived from.
    """
    required = {
        "blockchain", "auction_id", "solver",
        "sell_token", "buy_token", "failed_volume_native",
    }
    missing = sorted(required - set(volumes.columns))
    if missing:
        raise KeyError(f"Failed-volume CSVs are missing required columns: {missing}")

    failed = volumes.copy()
    for column in ["solver", "sell_token", "buy_token"]:
        failed[column] = failed[column].astype(str).str.lower().str.strip()
    failed["failed_volume_native"] = (
        pd.to_numeric(failed["failed_volume_native"], errors="coerce").fillna(0) / WEI
    )

    correlated = np.zeros(len(failed), dtype=bool)
    for chain, token_groups in groups_by_chain.items():
        on_chain = failed["blockchain"].eq(chain).to_numpy()
        for token_group in token_groups:
            correlated |= (
                on_chain
                & failed["sell_token"].isin(token_group).to_numpy()
                & failed["buy_token"].isin(token_group).to_numpy()
            )

    uncorrelated_rate = failed["blockchain"].map(UNCORRELATED_BPS_BY_CHAIN)
    if uncorrelated_rate.isna().any():
        unconfigured = sorted(failed.loc[uncorrelated_rate.isna(), "blockchain"].unique())
        raise ValueError(f"No uncorrelated-token rate configured for: {unconfigured}")

    rate_bps = np.where(correlated, CORRELATED_BPS, uncorrelated_rate.astype(float))
    failed["proposed_cap_native"] = rate_bps / 1e4 * failed["failed_volume_native"]

    return failed.groupby(AUCTION_KEYS, as_index=False).agg(
        proposed_cap_native=("proposed_cap_native", "sum"),
    )


def build_auctions(rewards, volumes, groups_by_chain):
    """One row per (blockchain, auction_id, solver).

    A scenario is a pair of columns, not a duplicated row:

      uncapped_native       signed reward before any cap (negative = penalty)
      current_native        signed reward under today's flat cap, as actually paid
      proposed_native       signed reward under the proposed volume-based cap
      proposed_cap_native   the proposed cap itself, from failed volume

    The performance reward is untouched by the proposal, so it is one column
    rather than two: only the penalty side is re-capped.
    """
    required = {
        "blockchain", "auction_id", "solver", "accounting_period",
        "is_excluded_from_penalties", "reward_penalty_native",
        "reward_penalty_uncapped_native", "reward_cap_upper_native",
    }
    missing = sorted(required - set(rewards.columns))
    if missing:
        raise KeyError(f"Reward CSVs are missing required columns: {missing}")

    auctions = rewards.rename(
        columns={
            "reward_penalty_native": "current_native",
            "reward_penalty_uncapped_native": "uncapped_native",
            "reward_cap_upper_native": "upper_reward_cap_native",
            "is_excluded_from_penalties": "excluded",
        }
    ).copy()

    auctions["solver"] = auctions["solver"].astype(str).str.lower().str.strip()
    auctions["accounting_period"] = auctions["accounting_period"].astype(str)
    for column in ["current_native", "uncapped_native", "upper_reward_cap_native"]:
        auctions[column] = pd.to_numeric(auctions[column], errors="coerce") / WEI
    auctions["excluded"] = (
        auctions["excluded"].astype(str).str.lower()
        .map({"true": True, "false": False}).fillna(False)
    )

    caps = proposed_penalty_caps(volumes, groups_by_chain)
    auctions = auctions.merge(caps, on=AUCTION_KEYS, how="left", validate="one_to_one")
    auctions["proposed_cap_native"] = auctions["proposed_cap_native"].fillna(0.0)

    # What each formula pays. The reward side is identical; only the penalty differs.
    auctions["performance_reward_native"] = auctions["current_native"].clip(lower=0)
    auctions["uncapped_penalty_native"] = (-auctions["uncapped_native"]).clip(lower=0)
    auctions["current_penalty_native"] = (-auctions["current_native"]).clip(lower=0)
    auctions["proposed_penalty_native"] = np.where(
        auctions["excluded"],
        0.0,
        np.minimum(auctions["uncapped_penalty_native"], auctions["proposed_cap_native"]),
    )
    auctions["proposed_native"] = (
        auctions["performance_reward_native"] - auctions["proposed_penalty_native"]
    )

    # Whatever the upper cap does not pay out becomes the consistency budget, so a
    # harsher penalty enlarges the pool that is then shared out again.
    for scenario in ["current", "proposed"]:
        budget = auctions["upper_reward_cap_native"] - auctions[f"{scenario}_native"]
        auctions[f"{scenario}_consistency_budget_native"] = budget
        if (budget < -1e-9).any():
            raise ValueError(
                f"Negative {scenario} consistency budget found:\n"
                + auctions.loc[budget < -1e-9, AUCTION_KEYS].head(20).to_string(index=False)
            )

    return auctions[
        AUCTION_KEYS
        + ["accounting_period", "excluded", "upper_reward_cap_native",
           "uncapped_native", "current_native", "proposed_native",
           "proposed_cap_native", "performance_reward_native",
           "uncapped_penalty_native", "current_penalty_native",
           "proposed_penalty_native", "current_consistency_budget_native",
           "proposed_consistency_budget_native"]
    ]


def prepare_consistency_shares(shares):
    shares = shares.copy()
    shares["solver"] = shares["solver"].astype(str).str.lower().str.strip()

    direct_share = pd.to_numeric(shares["consistency_reward_share"], errors="coerce")
    total_budget = pd.to_numeric(shares["total_consistency_budget"], errors="coerce")
    solver_reward = pd.to_numeric(shares["consistency_reward_native"], errors="coerce")
    calculated_share = solver_reward / total_budget.replace(0, np.nan)

    shares["consistency_reward_share"] = direct_share.fillna(calculated_share).fillna(0)

    return shares.groupby(PERIOD_KEYS, as_index=False).agg(
        consistency_reward_share=("consistency_reward_share", "first"),
    )


def calculate_solver_payments(auctions, shares):
    """One row per (blockchain, solver), both scenarios side by side."""
    required = {"blockchain", "accounting_period", "solver", "consistency_reward_share"}
    missing = sorted(required - set(shares.columns))
    if missing:
        raise KeyError(f"Consistency CSVs are missing columns: {missing}")

    share_data = prepare_consistency_shares(shares)
    period_keys = ["blockchain", "accounting_period"]

    weekly_budget = auctions.groupby(period_keys, as_index=False).agg(
        current_budget_native=("current_consistency_budget_native", "sum"),
        proposed_budget_native=("proposed_consistency_budget_native", "sum"),
    )

    share_sums = share_data.groupby(period_keys, as_index=False).agg(
        share_sum=("consistency_reward_share", "sum"),
    )
    share_check = weekly_budget.merge(share_sums, on=period_keys, how="left")
    bad_periods = share_check[
        share_check["current_budget_native"].abs().gt(1e-9)
        & ~np.isclose(share_check["share_sum"].fillna(0), 1.0, atol=1e-8)
    ]
    if not bad_periods.empty:
        raise ValueError(
            "Consistency shares do not sum to one:\n"
            + bad_periods[period_keys + ["share_sum"]].to_string(index=False)
        )

    allocated = share_data.merge(weekly_budget, on=period_keys, how="inner")
    for scenario in ["current", "proposed"]:
        allocated[f"{scenario}_consistency_native"] = (
            allocated["consistency_reward_share"] * allocated[f"{scenario}_budget_native"]
        )

    weekly_batch = auctions.groupby(PERIOD_KEYS, as_index=False).agg(
        performance_reward_native=("performance_reward_native", "sum"),
        current_penalty_native=("current_penalty_native", "sum"),
        proposed_penalty_native=("proposed_penalty_native", "sum"),
        current_batch_native=("current_native", "sum"),
        proposed_batch_native=("proposed_native", "sum"),
    )

    # outer: a solver can hold a consistency share in a period it won no auctions in
    payments = weekly_batch.merge(
        allocated[PERIOD_KEYS + ["current_consistency_native", "proposed_consistency_native"]],
        on=PERIOD_KEYS,
        how="outer",
    )
    value_columns = [
        "performance_reward_native", "current_penalty_native", "proposed_penalty_native",
        "current_batch_native", "proposed_batch_native",
        "current_consistency_native", "proposed_consistency_native",
    ]
    payments[value_columns] = payments[value_columns].fillna(0)

    for scenario in ["current", "proposed"]:
        payments[f"{scenario}_total_native"] = (
            payments[f"{scenario}_batch_native"] + payments[f"{scenario}_consistency_native"]
        )

    result = payments.groupby(["blockchain", "solver"], as_index=False)[
        value_columns + ["current_total_native", "proposed_total_native"]
    ].sum()
    result["change_native"] = result["proposed_total_native"] - result["current_total_native"]

    # Re-capping only moves payments between solvers: the pool is the upper cap either way.
    totals = result.groupby("blockchain")[
        ["current_total_native", "proposed_total_native"]
    ].sum()
    drift = (totals["proposed_total_native"] - totals["current_total_native"]).abs().max()
    assert drift < 1e-8, totals
    print("Total payments are unchanged across scenarios.")

    return result


def plot_solver_counterfactual(solver_payments, start_date, end_date):
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    for chain in sorted(solver_payments["blockchain"].dropna().unique()):
        chart_data = (
            solver_payments[solver_payments["blockchain"].eq(chain)]
            .sort_values("current_total_native", ascending=False)
        )
        width = max(1100, 30 * len(chart_data) + 300)
        fig = go.Figure()
        fig.add_bar(x=chart_data["solver"], y=chart_data["current_total_native"], name="Current")
        fig.add_bar(x=chart_data["solver"], y=chart_data["proposed_total_native"], name="Proposed")
        fig.add_hline(y=0, line_width=0.8)
        fig.update_xaxes(title_text="Solver address", tickangle=-90)
        fig.update_layout(
            title=(
                "Solvers PnL Counterfactual — Proposed Penalty Caps — "
                f"{chain} — {start} to {end}"
            ),
            yaxis_title="Total payment (native token)",
            barmode="group",
            template="plotly_white",
            width=width,
            height=700,
            legend_title_text="Scenario",
        )
        fig.show()


def run_counterfactual(
    start_date=START_DATE,
    end_date=END_DATE,
    data_dir=DATA_DIR,
    chains=None,
    fetch_script=FETCH_SCRIPT,
    make_plots=True,
):
    if chains is None:
        chains = list(CHAIN_ALIASES)
    fetch_inputs_if_missing(
        start_date=start_date, end_date=end_date, chains=chains,
        data_dir=data_dir, fetch_script=fetch_script,
    )
    rewards, volumes, shares = load_inputs(
        data_dir=data_dir, start_date=start_date, end_date=end_date, chains=chains,
    )

    chains = sorted(rewards["blockchain"].dropna().unique())
    unknown_chains = sorted(set(chains) - set(CHAIN_ALIASES))
    if unknown_chains:
        raise ValueError(f"Missing CHAIN_ALIASES entries for: {unknown_chains}")

    correlated_groups = load_correlated_groups(chains)
    auctions = build_auctions(rewards, volumes, correlated_groups)
    solver_payments = calculate_solver_payments(auctions, shares)

    for chain in chains:
        table = (
            solver_payments[solver_payments["blockchain"].eq(chain)]
            .sort_values("current_total_native", ascending=False)
        )
        rate = UNCORRELATED_BPS_BY_CHAIN[chain]
        print(
            f"\n=== {chain} | uncorrelated={rate:g} bps, "
            f"correlated={CORRELATED_BPS:g} bps ==="
        )
        display(
            table[
                ["solver", "performance_reward_native",
                 "current_penalty_native", "proposed_penalty_native",
                 "current_consistency_native", "proposed_consistency_native",
                 "current_total_native", "proposed_total_native", "change_native"]
            ].round(6)
        )

    if make_plots:
        plot_solver_counterfactual(solver_payments, start_date, end_date)

    return solver_payments, auctions


SOLVER_PAYMENTS, AUCTIONS = run_counterfactual()
